# Gathering Middle Frames and Audio from Audiocaps2.0 Dataset

In [ ]:
# !pip install imageio-ffmpeg

In [ ]:
# load packages
import pandas as pd
import numpy as np
import random
import os
import imageio.v3 as iio
import shutil
import wave

## Prepare Data

In [ ]:
# load audiocaps csv downloaded from audiocaps github repo
audiocaps_test = pd.read_csv("/content/Audiocaps2.0/test.csv")
audiocaps_train = pd.read_csv("/content/Audiocaps2.0/train.csv")
audiocaps_val = pd.read_csv("/content/Audiocaps2.0/val.csv")

audiocaps = pd.concat([audiocaps_test, audiocaps_train, audiocaps_val], ignore_index=True)
audiocaps.shape

(98607, 4)

In [ ]:
# compile audiocap ids for audio and video files using youtube id filenames

### audio files
audio_folder = "/content/audiocaps_raw_audio"
audio_files = [f for f in os.listdir(audio_folder) if f.endswith(".wav")]
print("Total number of audio files:", len(audio_files))

audio_ids = []

for f in audio_files:
    file_path = os.path.join(audio_folder, f)

    # skip if audio is zero bytes
    if os.path.getsize(file_path) == 0:
        print("ZERO BYTES:", f)
        continue

    # remove start time from file name --> get youtube id
    youtube_id = f.rsplit("_", 1)[0]

    row = audiocaps.loc[audiocaps["youtube_id"] == youtube_id]

    # skip audio not in audiocaps2.0 dataset
    if row.empty:
        continue

    audiocap_id = row["audiocap_id"].iloc[0]

    audio_ids.append(audiocap_id)

print("Number of audio files in Audiocaps2.0:", len(audio_ids))


### video files
video_folder = "/content/audiocaps_raw_video"

video_files = [f for f in os.listdir(video_folder) if f.endswith(".mkv")]
print("Total number of video files:", len(video_files))

video_ids = []

for f in video_files:
    file_path = os.path.join(video_folder, f)

    # skip if video is zero bytes
    if os.path.getsize(file_path) == 0:
        print("ZERO BYTES:", f)
        continue

    # remove start time from file name --> get youtube id
    youtube_id = f.rsplit("_", 1)[0]

    row = audiocaps.loc[audiocaps["youtube_id"] == youtube_id]

    # skip videos not in audiocaps2.0 dataset
    if row.empty:
        # print("Not in Audiocaps2.0:", f)
        continue

    audiocap_id = row["audiocap_id"].iloc[0]

    video_ids.append(audiocap_id)

print("Number of non-zero video files in Audiocaps2.0:", len(video_ids))

Total number of audio files: 203388
Number of audio files in Audiocaps2.0: 92724
Total number of video files: 100817
ZERO BYTES: lUL5N-RdJl0_150.mkv
ZERO BYTES: 5hVgho0Z9V0_100.mkv
ZERO BYTES: 5yWDAqqzddo_230.mkv
ZERO BYTES: 5Seyx8gJxSk_10.mkv
ZERO BYTES: pY5l0Iv_9sU_140.mkv
ZERO BYTES: 5_Ieqp54X0c_510.mkv
Number of non-zero video files in Audiocaps2.0: 92724


In [ ]:
# indicator variables for available audio or video files
audiocaps["audio"] = audiocaps["audiocap_id"].isin(audio_ids).astype(int)
audiocaps["video"] = audiocaps["audiocap_id"].isin(video_ids).astype(int)

# save audiocaps before filtering
audiocaps.to_csv("/content/Audiocaps2.0/all_audiocaps.csv")

# filter for available audio and video files
df_filtered = audiocaps[(audiocaps["audio"] == 1) & (audiocaps["video"] == 1)]

# save filtered dataframe as .csv
df_filtered.to_csv("/content/Audiocaps2.0/audiocaps_vid_audio.csv")

print(df_filtered.shape)
df_filtered.head()

(92724, 6)


,audiocap_id,youtube_id,start_time,caption,audio,video
0,103549,7fmOlUlwoNg,20.0,constant rattling noise and sharp vibrations,1,1
1,103548,6BJ455B1aAs,0.0,a rocket flies by followed by a loud explosion...,1,1
2,103541,GOD8Bt5LfDE,100.0,humming and vibrating with a man and children ...,1,1
3,103540,YQSuFyFm3Lc,230.0,a train running on a railroad track followed b...,1,1
4,103542,VjSEIRnLAh8,30.0,"food is frying, and a woman talks",1,1


## Extract Center Frames in Video Files

In [ ]:
input_video = "/content/audiocaps_raw_video"
output_frames = "/content/middle_frames"
os.makedirs(output_frames, exist_ok=True)

filtered_ids = df_filtered["audiocap_id"].tolist()

# randomly select audiocap ids
random.seed(42)
sampled_ids = random.sample(filtered_ids, 800) # previously 650

for audiocap_id in sampled_ids:
    output_path = os.path.join(output_frames, f"{audiocap_id}.png")

    if os.path.exists(output_path):
        continue

    row = df_filtered.loc[df_filtered["audiocap_id"] == audiocap_id].iloc[0]
    video_file = f"{row['youtube_id']}_{int(row['start_time'])}.mkv"
    input_path = os.path.join(input_video, video_file)

    frames = list(iio.imiter(input_path))
    middle_index = len(frames) // 2
    iio.imwrite(output_path, frames[middle_index])

After extracting the middle frames from video files, manually filter to remove inadequate images. The decision were based on the following criteria:

- No blank images (e.g. black screen)
- No text on screen
- No video game screenshots
- No animation screenshots

## Gather audio files with corresponding filtered image frames

In [ ]:
# audio folders
input_audio = "/content/audiocaps_raw_audio"
output_audio = "/content/audio"
os.makedirs(output_audio, exist_ok=True)

filtered_ids = [f.removesuffix(".png") for f in os.listdir(output_frames) if f.endswith(".png")]

for audiocap_id in filtered_ids:
    output_path = os.path.join(output_audio, f"{audiocap_id}.wav")

    if os.path.exists(output_path):
        continue

    row = df_filtered.loc[df_filtered["audiocap_id"].astype(str) == audiocap_id].iloc[0]
    audio_file = f"{row['youtube_id']}_{int(row['start_time'])}.wav"
    input_path = os.path.join(input_audio, audio_file)

    shutil.copy(input_path, output_path)

In [145]:
def get_wav_duration(file_path):
    with wave.open(file_path, 'r') as wav_file:
        frames = wav_file.getnframes()
        rate = wav_file.getframerate()
        duration = frames / float(rate)
    return duration

In [ ]:
aud_ids = sorted([f for f in os.listdir(output_audio) if f.endswith(".wav")])

duration_dict = {}

for f in aud_ids:
    # print(f"Duration: {get_wav_duration(os.path.join(output_audio, f))} seconds")
    duration_dict[f] = get_wav_duration(os.path.join(output_audio, f))

aud_durations = pd.DataFrame({'audio_file': duration_dict.keys(), 'duration': duration_dict.values()})

aud_durations['duration'].mean()

In [ ]:
aud_durations[aud_durations['duration'] < 9]

Manually remove audio files less than 9 seconds (and corresponding middle frame)

## Final Dataset of Selected Middle Frames and Audio

In [170]:
# check that frames and audio match
vid_ids = sorted([f.removesuffix(".png") for f in os.listdir(output_frames) if f.endswith(".png")])
aud_ids = sorted([f.removesuffix(".wav") for f in os.listdir(output_audio) if f.endswith(".wav")])

vid_ids == aud_ids

True

In [ ]:
df_selected = df_filtered[df_filtered["audiocap_id"].astype(str).isin(vid_ids)]
print("Number of selected samples:", df_selected.shape[0])

# save as .csv file
df_selected.to_csv("/content/final_audiocaps.csv")

Number of selected samples: 620
